<a href="https://colab.research.google.com/github/Project-MANAS-Research-AI/Jansi_Bapodara_Research_AI/blob/main/causal_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch torchvision torchaudio transformer-lens matplotlib

In [ ]:
import torch
print(torch.__version__)

In [ ]:
pip uninstall torch torchvision torchaudio

In [ ]:
pip install torch torchvision torchaudio

In [ ]:
import torch
import torchaudio

print(torch.__version__)

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from transformer_lens import HookedTransformer

In [ ]:
model = HookedTransformer.from_pretrained("gpt2-small")
model.eval()

In [ ]:
clean_prompt = "The capital of France is"
corrupt_prompt = "The capital of India is"

In [ ]:
tokens_clean = model.to_tokens(clean_prompt)
tokens_corrupt = model.to_tokens(corrupt_prompt)

In [ ]:
print(model.to_str_tokens(tokens_clean))
print(model.to_str_tokens(tokens_corrupt))

In [ ]:
clean_logits,clean_cache= model.run_with_cache(clean_prompt)
clean_cache.keys()

In [ ]:
corrupt_logits,corrupt_cache= model.run_with_cache(corrupt_prompt)
corrupt_cache.keys()

In [ ]:
answer_token = model.to_single_token(" Paris")

print(answer_token)

In [ ]:
clean_answer_logit = clean_logits[0, -1, answer_token]
corrupt_answer_logit = corrupt_logits[0, -1, answer_token]

print("Clean:", clean_answer_logit.item())
print("Corrupt:", corrupt_answer_logit.item())

In [ ]:
import torch

def test_prompt(prompt, answer, model):

    # tokenize prompt
    tokens = model.to_tokens(prompt)

    token_strings = model.to_str_tokens(tokens)

    print("Tokenized prompt:", token_strings)


    # tokenize answer
    answer_tokens = model.to_tokens(answer, prepend_bos=False)

    answer_str = model.to_str_tokens(answer_tokens)

    print("Tokenized answer:", answer_str)


    # run model
    logits = model(tokens)

    # last token prediction
    last_logits = logits[0, -1, :]

    probs = torch.softmax(last_logits, dim=-1)


    # answer token id
    answer_token_id = answer_tokens[0,0].item()


    print("\nPerformance on answer token:")

    print(
        f"Logit: {last_logits[answer_token_id]:.2f} "
        f"Prob: {probs[answer_token_id]*100:.2f}% "
        f"Token: |{answer}|"
    )


    # top predictions
    top_logits, top_tokens = torch.topk(last_logits, 10)


    print("\nTop predictions:")


    for i, (logit, token_id) in enumerate(zip(top_logits, top_tokens)):

        token_id = token_id.item()

        token = model.to_single_str_token(token_id)

        probability = probs[token_id]

        print(
            f"{i}th token. "
            f"Logit: {logit.item():.2f} "
            f"Prob: {probability.item()*100:.2f}% "
            f"Token: |{token}|"
        )


    print("\nRank of answer token:")

    rank = (last_logits > last_logits[answer_token_id]).sum().item() + 1

    print([(answer, rank)])

In [ ]:
test_prompt(
    "The capital of France is",
    " Paris",
    model
)

In [ ]:
prompt = "The capital city of France is called"

logits = model(prompt)

top_logits, top_tokens = torch.topk(logits[0,-1], 10)

for logit, token in zip(top_logits, top_tokens):
    print(
        model.to_single_str_token(token.item()),
        logit.item()
    )

The model required completion and clarity in  the sentence. We can also see that it gives better probability for the last prompt therefore I have changed the clean and corrupt prompt to "The capital city of ____ is called"

In [ ]:
clean_prompt = "The capital city of France is called"
corrupt_prompt = "The capital city of India is called"

In [ ]:
tokens_clean = model.to_tokens(clean_prompt)
tokens_corrupt = model.to_tokens(corrupt_prompt)
print(model.to_str_tokens(tokens_clean))
print(model.to_str_tokens(tokens_corrupt))

In [ ]:
clean_logits,clean_cache= model.run_with_cache(clean_prompt)
clean_cache.keys()
corrupt_logits,corrupt_cache= model.run_with_cache(corrupt_prompt)
corrupt_cache.keys()

In [ ]:
answer_token = model.to_single_token(" Paris")

print(answer_token)
clean_answer_logit = clean_logits[0, -1, answer_token]
corrupt_answer_logit = corrupt_logits[0, -1, answer_token]

print("Clean:", clean_answer_logit.item())
print("Corrupt:", corrupt_answer_logit.item())

In [ ]:
position = 5


def patch_activation(activation, hook):

    activation[:, position, :] = clean_cache[hook.name][:, position, :]

    return activation

In [ ]:
patched_logits = model.run_with_hooks(
    corrupt_prompt,
    fwd_hooks=[
        (
            "blocks.0.hook_resid_post",
            patch_activation
        )
    ]
)

In [ ]:
patched_answer_logit = patched_logits[0, -1, answer_token]

print("Clean:", clean_answer_logit.item())
print("Corrupt:", corrupt_answer_logit.item())
print("Patched:", patched_answer_logit.item())


In [ ]:
print(model.to_str_tokens(tokens_clean))
print(model.cfg.n_layers)

In [ ]:
n_layers = model.cfg.n_layers
seq_len = tokens_clean.shape[1]

recovery = torch.zeros(n_layers, seq_len)

print(recovery.shape)

In [ ]:
with torch.no_grad():

    for layer in range(n_layers):

        for patch_position in range(seq_len):

            patched_logits = model.run_with_hooks(
                corrupt_prompt,
                fwd_hooks=[
                    (
                        f"blocks.{layer}.hook_resid_post",
                        patch_activation
                    )
                ]
            )

            patched_answer_logit = patched_logits[0, -1, answer_token].item()

            recovery[layer, patch_position] = (
                patched_answer_logit - corrupt_answer_logit.item()
            ) / (
                clean_answer_logit.item() - corrupt_answer_logit.item()
            )

            model.reset_hooks()

In [ ]:
print(recovery)

In [ ]:
plt.figure(figsize=(10,6))

plt.imshow(
    recovery.detach().numpy(),
    aspect="auto",
    cmap = 'magma'
)

plt.xlabel("Token Position")
plt.ylabel("Layer")
plt.colorbar()

plt.show()

Now since we have a basic layout for 1 sentence, let's explore for more prompts

In [ ]:
prompt_pairs = [

    (
        "The author of Harry Potter book series is J. K.",
        "The author of Harry Williams book series is J. K.",
        " Rowling"
    ),

    (
        "The planet known as the Red Planet is",
        "The planet known as the blue Planet is",
        " Mars"
    ),

    (
        "The first month of the year is",
        "The last month of the year is",
        " January"
    ),

    (
        "The currency of USA is the",
        "The currency of Japan is the",
        " Dollar"
    )

]


In [ ]:
answer = model.to_str_tokens(" dollar")
print(answer)

In [ ]:
def run_patching(clean_prompt, corrupt_prompt, answer):

    tokens_clean = model.to_tokens(clean_prompt)

    clean_logits, clean_cache = model.run_with_cache(
        clean_prompt
    )

    corrupt_logits, corrupt_cache = model.run_with_cache(
        corrupt_prompt
    )


    answer_token = model.to_single_token(answer)


    clean_answer_logit = clean_logits[0,-1,answer_token]

    corrupt_answer_logit = corrupt_logits[0,-1,answer_token]


    del clean_logits
    del corrupt_logits


    n_layers = model.cfg.n_layers
    seq_len = tokens_clean.shape[1]


    recovery = torch.zeros(
        n_layers,
        seq_len
    )


    def patch_activation(activation, hook):

        activation[:, patch_position, :] = (
            clean_cache[hook.name][:, patch_position, :]
        )

        return activation



    with torch.no_grad():

        for layer in range(n_layers):

            for patch_position in range(seq_len):


                patched_logits = model.run_with_hooks(

                    corrupt_prompt,

                    fwd_hooks=[
                        (
                            f"blocks.{layer}.hook_resid_post",
                            patch_activation
                        )
                    ]
                )


                patched_answer_logit = (
                    patched_logits[0,-1,answer_token]
                )


                recovery[layer, patch_position] = (

                    patched_answer_logit - corrupt_answer_logit

                ) / (

                    clean_answer_logit - corrupt_answer_logit

                )


                model.reset_hooks()


                del patched_logits


    del clean_cache
    del corrupt_cache

    torch.cuda.empty_cache()


    return recovery

In [ ]:
all_recoveries = []


for clean_prompt, corrupt_prompt, answer in prompt_pairs:


    print("Running:", clean_prompt)


    result = run_patching(
        clean_prompt,
        corrupt_prompt,
        answer
    )


    all_recoveries.append(result)


    del result

    torch.cuda.empty_cache()

In [ ]:
max_len = max(
    r.shape[1] for r in all_recoveries
)


padded_recoveries = []


for r in all_recoveries:

    pad_size = max_len - r.shape[1]

    padded = torch.nn.functional.pad(
        r,
        (0, pad_size)
    )

    padded_recoveries.append(padded)



average_recovery = torch.stack(
    padded_recoveries
).mean(dim=0)


print(average_recovery.shape)

In [ ]:
plt.figure(figsize=(10,6))

plt.imshow(
    average_recovery.numpy(),
    aspect="auto",
    cmap= 'coolwarm'
)

plt.xlabel("Token Position")

plt.ylabel("Layer")

plt.colorbar(
    label="Recovery"
)

plt.show()

In [ ]:
prompt_pairs = [

    (
        "The capital city of India is called",
        "The capital city of USA is called",
        " Delhi"
    ),

    (
        "The capital city of Italy is called",
        "The capital city of USA is called",
        " Rome"
    ),

    (
        "The capital city of Japan is called",
        "The capital city of India is called",
        " Tokyo"
    )

]

In [ ]:
answer = model.to_str_tokens(" Delhi")
print(answer)

In [ ]:
def run_patching(clean_prompt, corrupt_prompt, answer):

    tokens_clean = model.to_tokens(clean_prompt)

    clean_logits, clean_cache = model.run_with_cache(
        clean_prompt
    )

    corrupt_logits, corrupt_cache = model.run_with_cache(
        corrupt_prompt
    )


    answer_token = model.to_single_token(answer)


    clean_answer_logit = clean_logits[0,-1,answer_token]

    corrupt_answer_logit = corrupt_logits[0,-1,answer_token]


    del clean_logits
    del corrupt_logits


    n_layers = model.cfg.n_layers
    seq_len = tokens_clean.shape[1]


    recovery = torch.zeros(
        n_layers,
        seq_len
    )


    def patch_activation(activation, hook):

        activation[:, patch_position, :] = (
            clean_cache[hook.name][:, patch_position, :]
        )

        return activation



    with torch.no_grad():

        for layer in range(n_layers):

            for patch_position in range(seq_len):


                patched_logits = model.run_with_hooks(

                    corrupt_prompt,

                    fwd_hooks=[
                        (
                            f"blocks.{layer}.hook_resid_post",
                            patch_activation
                        )
                    ]
                )


                patched_answer_logit = (
                    patched_logits[0,-1,answer_token]
                )


                recovery[layer, patch_position] = (

                    patched_answer_logit - corrupt_answer_logit

                ) / (

                    clean_answer_logit - corrupt_answer_logit

                )


                model.reset_hooks()


                del patched_logits


    del clean_cache
    del corrupt_cache

    torch.cuda.empty_cache()


    return recovery

In [ ]:
all_recoveries = []


for clean_prompt, corrupt_prompt, answer in prompt_pairs:


    print("Running:", clean_prompt)


    result = run_patching(
        clean_prompt,
        corrupt_prompt,
        answer
    )


    all_recoveries.append(result)


    del result

    torch.cuda.empty_cache()

In [ ]:
average_recovery = torch.stack(
    all_recoveries
).mean(dim=0)


print(average_recovery.shape)

In [ ]:
plt.figure(figsize=(10,6))

plt.imshow(
    average_recovery.numpy(),
    aspect="auto"
)

plt.xlabel("Token Position")

plt.ylabel("Layer")

plt.colorbar(
    label="Recovery"
)

plt.show()

In [ ]:
prompt_pairs = [

    (
        "The planet known as the Red Planet is",
        "The planet known as the blue Planet is",
        " Mars"
    ),

    (
        "The planet known as the largest Planet is",
        "The planet known as the smallest Planet is",
        " Jupiter"
    )

]

In [ ]:
def run_patching(clean_prompt, corrupt_prompt, answer):

    tokens_clean = model.to_tokens(clean_prompt)

    clean_logits, clean_cache = model.run_with_cache(
        clean_prompt
    )

    corrupt_logits, corrupt_cache = model.run_with_cache(
        corrupt_prompt
    )


    answer_token = model.to_single_token(answer)


    clean_answer_logit = clean_logits[0,-1,answer_token]

    corrupt_answer_logit = corrupt_logits[0,-1,answer_token]


    del clean_logits
    del corrupt_logits


    n_layers = model.cfg.n_layers
    seq_len = tokens_clean.shape[1]


    recovery = torch.zeros(
        n_layers,
        seq_len
    )


    def patch_activation(activation, hook):

        activation[:, patch_position, :] = (
            clean_cache[hook.name][:, patch_position, :]
        )

        return activation



    with torch.no_grad():

        for layer in range(n_layers):

            for patch_position in range(seq_len):


                patched_logits = model.run_with_hooks(

                    corrupt_prompt,

                    fwd_hooks=[
                        (
                            f"blocks.{layer}.hook_resid_post",
                            patch_activation
                        )
                    ]
                )


                patched_answer_logit = (
                    patched_logits[0,-1,answer_token]
                )


                recovery[layer, patch_position] = (

                    patched_answer_logit - corrupt_answer_logit

                ) / (

                    clean_answer_logit - corrupt_answer_logit

                )


                model.reset_hooks()


                del patched_logits


    del clean_cache
    del corrupt_cache

    torch.cuda.empty_cache()


    return recovery

In [ ]:
all_recoveries = []


for clean_prompt, corrupt_prompt, answer in prompt_pairs:


    print("Running:", clean_prompt)


    result = run_patching(
        clean_prompt,
        corrupt_prompt,
        answer
    )


    all_recoveries.append(result)


    del result

    torch.cuda.empty_cache()

In [ ]:
average_recovery = torch.stack(
    all_recoveries
).mean(dim=0)


print(average_recovery.shape)

In [ ]:
plt.figure(figsize=(10,6))

plt.imshow(
    average_recovery.numpy(),
    aspect="auto",
    cmap= 'icefire'
)

plt.xlabel("Token Position")

plt.ylabel("Layer")

plt.colorbar(
    label="Recovery"
)

plt.show()